In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# STEP 1: Load Dataset
df = pd.read_csv("loan_data.csv")

print("Original Data")
print(df.head())

print("\nShape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()

# STEP 2: Separate Features and Target
df = df.dropna(subset=["loan_approved"])

X = df.drop(columns=["loan_approved"])
y = df["loan_approved"].astype(int)

# STEP 3: Identify Numerical and Categorical Columns
num = X.select_dtypes(include="number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()

# STEP 4: Numerical Preprocessing
numeric = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# STEP 5: Categorical Preprocessing
categorical = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocess = ColumnTransformer([
    ("numeric", numeric, num),
    ("categorical", categorical, cat)
])

# STEP 6: Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# STEP 7: All Four Algorithms
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="rbf"),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

# STEP 8: Train and Evaluate
results = []

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocessing", preprocess),
        ("classifier", algorithm)
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("\n==============================")
    print("Model:", name)
    print("==============================")

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        zero_division=0
    ))

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

# STEP 9: Compare All Four Models
comparison = pd.DataFrame(results)

print("\nFinal Model Comparison:")
print(comparison.round(4))

# STEP 10: Find Best Model by Accuracy
best = comparison.loc[comparison["Accuracy"].idxmax()]

print("\nBest Model by Accuracy:", best["Model"])
print("Accuracy:", round(best["Accuracy"] * 100, 2), "%")

FileNotFoundError: [Errno 2] No such file or directory: 'loan_data.csv'